# **Feature Engineering e Pipeline de Pré-processamento**

**1.** **Bibliotecas**

In [22]:
import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("✓ Bibliotecas importadas com sucesso.")

✓ Bibliotecas importadas com sucesso.


**2. Configurações**

In [23]:
TARGET = "alfabetizado"
PESO = "peso_aluno"
GRUPO_MUNICIPIO = "id_municipio"

RANDOM_STATE = 42

print("✓ Configurações definidas.")

✓ Configurações definidas.


**3. Carregamento da base**

In [24]:
CAMINHO_BASE = "base_analitica_amostra.parquet"

df = pd.read_parquet(CAMINHO_BASE)

print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")
df.head()

Linhas: 296,292
Colunas: 26


,id_aluno,id_escola,id_municipio,rede,caderno,peso_aluno,nome_municipio,sigla_uf,nome_regiao,nome_mesorregiao,...,va_servicos,va_adespss,taxa_municipio_2023,media_portugues_municipio_2023,meta_alfabetizacao_2024,meta_alfabetizacao_2026,meta_alfabetizacao_2030,nivel_alfabetizacao,percentual_participacao,alfabetizado
0,12009423,60000594,1200401,2,20,1.11,Rio Branco,AC,Norte,Vale do Acre,...,5153657000,2949436000,NaN,NaN,NaN,64.86,80.0,2,81.03,0
1,12008971,60000525,1200401,2,7,1.03,Rio Branco,AC,Norte,Vale do Acre,...,5153657000,2949436000,NaN,NaN,NaN,64.86,80.0,2,81.03,0
2,12003465,60000436,1200328,2,15,1.36,Jordão,AC,Norte,Vale do Juruá,...,14031000,73684000,NaN,NaN,NaN,38.39,80.0,0,93.88,0
3,12005920,60000409,1200609,2,17,1.40,Tarauacá,AC,Norte,Vale do Juruá,...,158432000,322737000,NaN,NaN,NaN,58.25,80.0,1,85.99,0
4,12005072,60000526,1200450,2,13,1.14,Senador Guiomard,AC,Norte,Vale do Acre,...,150418000,175830000,NaN,NaN,NaN,57.26,80.0,1,76.67,0


**4. Auditoria inicial**

In [25]:
auditoria = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "%_nulos": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique()
})

auditoria

print("✓ Auditoria inicial concluída.")

✓ Auditoria inicial concluída.


**5. Target, peso e grupos**

In [26]:
y = df[TARGET]

sample_weight = df[PESO]

groups = df[GRUPO_MUNICIPIO]

print("✓ Target, peso amostral e grupos separados.")

✓ Target, peso amostral e grupos separados.


**6. Remoção de colunas que não serão features**

In [27]:
COLUNAS_EXCLUIDAS = [
    "alfabetizado",
    "peso_aluno",
    "id_aluno",
    "id_escola",
    "id_municipio",
    "nome_municipio"
]

X = df.drop(
    columns=COLUNAS_EXCLUIDAS
).copy()

print(f"Quantidade de features: {X.shape[1]}")

Quantidade de features: 20


**7. Verificação de leakage**

In [28]:
colunas_proibidas = [
    TARGET,
    PESO,
    "id_aluno",
    "id_escola",
    "id_municipio",
    "nome_municipio"
]

leakage = [
    coluna
    for coluna in colunas_proibidas
    if coluna in X.columns
]

if leakage:
    raise ValueError(
        f"Possível leakage: {leakage}"
    )

print("OK - Nenhuma coluna proibida nas features.")

OK - Nenhuma coluna proibida nas features.


**8. Feature Engineering**

In [30]:
class FeatureEngineer(
    BaseEstimator,
    TransformerMixin
):

    def fit(self, X, y=None):
        return self


    def transform(self, X):

        X = X.copy()

        populacao = X["populacao"].replace(
            0,
            np.nan
        )

        pib = X["pib"].replace(
            0,
            np.nan
        )

        # PIB per capita
        X["pib_per_capita"] = (
            X["pib"] / populacao
        )

        # Participação setorial no PIB
        X["pct_agropecuaria_pib"] = (
            X["va_agropecuaria"] / pib
        )

        X["pct_industria_pib"] = (
            X["va_industria"] / pib
        )

        X["pct_servicos_pib"] = (
            X["va_servicos"] / pib
        )

        X["pct_adespss_pib"] = (
            X["va_adespss"] / pib
        )

        # Indicadores de ausência de histórico
        X["sem_historico_taxa_2023"] = (
            X["taxa_municipio_2023"]
            .isna()
            .astype(int)
        )

        X["sem_historico_media_portugues_2023"] = (
            X["media_portugues_municipio_2023"]
            .isna()
            .astype(int)
        )

        # Existência de meta 2030
        X["tem_meta_2030"] = (
            X["meta_alfabetizacao_2030"]
            .notna()
            .astype(int)
        )

        return X

# Criar o transformador
feature_engineer = FeatureEngineer()

# Aplicar Feature Engineering
X_teste_fe = feature_engineer.fit_transform(X)

# Mensagens de validação
print("✓ Feature Engineering executada com sucesso.")
print(f"Features antes: {X.shape[1]}")
print(f"Features depois: {X_teste_fe.shape[1]}")
print(f"Novas features criadas: {X_teste_fe.shape[1] - X.shape[1]}")

✓ Feature Engineering executada com sucesso.
Features antes: 20
Features depois: 28
Novas features criadas: 8


**9. Features categóricas**

In [31]:
CATEGORICAL_FEATURES = [
    "rede",
    "caderno",
    "sigla_uf",
    "nome_regiao",
    "nome_mesorregiao"
]

print(
    f"✓ {len(CATEGORICAL_FEATURES)} features categóricas definidas."
)

✓ 5 features categóricas definidas.


**10. Features numéricas**

In [32]:
NUMERIC_FEATURES = [

    "capital_uf",
    "amazonia_legal",

    "populacao",
    "pib",
    "va_agropecuaria",
    "va_industria",
    "va_servicos",
    "va_adespss",

    "taxa_municipio_2023",
    "media_portugues_municipio_2023",

    "meta_alfabetizacao_2024",
    "meta_alfabetizacao_2026",

    "nivel_alfabetizacao",
    "percentual_participacao",

    # Feature Engineering
    "pib_per_capita",
    "pct_agropecuaria_pib",
    "pct_industria_pib",
    "pct_servicos_pib",
    "pct_adespss_pib",
    "sem_historico_taxa_2023",
    "sem_historico_media_portugues_2023",
    "tem_meta_2030"
]

print(
    f"✓ {len(NUMERIC_FEATURES)} features numéricas definidas."
)

✓ 22 features numéricas definidas.


**11. Pipeline numérica**

In [33]:
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

print("✓ Pipeline numérica criada.")

✓ Pipeline numérica criada.


**12. Pipeline categórica**

In [34]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

print("✓ Pipeline categórica criada.")

✓ Pipeline categórica criada.


**13. ColumnTransformer**

In [35]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            NUMERIC_FEATURES
        ),
        (
            "categorical",
            categorical_transformer,
            CATEGORICAL_FEATURES
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print("✓ ColumnTransformer criado com sucesso.")

✓ ColumnTransformer criado com sucesso.


**14. Pipeline final**

In [36]:
pipeline_preprocessamento = Pipeline(
    steps=[
        (
            "feature_engineering",
            FeatureEngineer()
        ),
        (
            "preprocessor",
            preprocessor
        )
    ]
)

print("✓ Pipeline de pré-processamento criada com sucesso.")

✓ Pipeline de pré-processamento criada com sucesso.


**15. Teste da pipeline**

In [37]:
X_processado = (
    pipeline_preprocessamento
    .fit_transform(X)
)

print(
    f"Shape original: {X.shape}"
)

print(
    f"Shape processado: {X_processado.shape}"
)

Shape original: (296292, 20)
Shape processado: (296292, 213)


**16. Features finais**

In [38]:

nomes_features = (
    pipeline_preprocessamento
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

print(
    f"Quantidade final de features: "
    f"{len(nomes_features)}"
)
pd.DataFrame({
    "feature": nomes_features
}).head(20)

Quantidade final de features: 213


,feature
0,capital_uf
1,amazonia_legal
2,populacao
3,pib
4,va_agropecuaria
5,va_industria
6,va_servicos
7,va_adespss
8,taxa_municipio_2023
9,media_portugues_municipio_2023


# **Entrega**

In [40]:
print("=" * 50)
print("CHECK FINAL DA PIPELINE")
print("=" * 50)

print("✓ Base carregada")
print("✓ Target separado")
print("✓ Peso amostral separado")
print("✓ Grupos definidos")
print("✓ Colunas de identificação removidas")
print("✓ Data Leakage verificado")
print("✓ Feature Engineering aplicada")
print("✓ Variáveis numéricas configuradas")
print("✓ Variáveis categóricas configuradas")
print("✓ ColumnTransformer criado")
print("✓ Pipeline executada com sucesso")

print("=" * 50)
print("PIPELINE PRONTA PARA A ETAPA DE MODELAGEM")
print("=" * 50)

CHECK FINAL DA PIPELINE
✓ Base carregada
✓ Target separado
✓ Peso amostral separado
✓ Grupos definidos
✓ Colunas de identificação removidas
✓ Data Leakage verificado
✓ Feature Engineering aplicada
✓ Variáveis numéricas configuradas
✓ Variáveis categóricas configuradas
✓ ColumnTransformer criado
✓ Pipeline executada com sucesso
PIPELINE PRONTA PARA A ETAPA DE MODELAGEM
